In [4]:
import requests

url = "https://api-web.nhle.com/v1/standings/now"
response = requests.get(url, timeout=10)
response.raise_for_status()
data = response.json()

team_records = []
for team in data.get("standings", []):
    team_abbrev = team.get("teamAbbrev", {})
    team_name = team.get("teamName", {})
    team_records.append({
        "division_name": team.get("divisionName", ""),
        "conference_name": team.get("conferenceName", ""),
        "team_name": team_name.get("default", "") if isinstance(team_name, dict) else team_name,
        "team_abbrev": team_abbrev.get("default", "") if isinstance(team_abbrev, dict) else team_abbrev,
        "logo_url": team.get("teamLogo"),
    })

team_records

[{'division_name': 'Central',
  'conference_name': 'Western',
  'team_name': 'Colorado Avalanche',
  'team_abbrev': 'COL',
  'logo_url': 'https://assets.nhle.com/logos/nhl/svg/COL_light.svg'},
 {'division_name': 'Metropolitan',
  'conference_name': 'Eastern',
  'team_name': 'Carolina Hurricanes',
  'team_abbrev': 'CAR',
  'logo_url': 'https://assets.nhle.com/logos/nhl/svg/CAR_light.svg'},
 {'division_name': 'Central',
  'conference_name': 'Western',
  'team_name': 'Dallas Stars',
  'team_abbrev': 'DAL',
  'logo_url': 'https://assets.nhle.com/logos/nhl/svg/DAL_light.svg'},
 {'division_name': 'Atlantic',
  'conference_name': 'Eastern',
  'team_name': 'Buffalo Sabres',
  'team_abbrev': 'BUF',
  'logo_url': 'https://assets.nhle.com/logos/nhl/svg/BUF_light.svg'},
 {'division_name': 'Atlantic',
  'conference_name': 'Eastern',
  'team_name': 'Tampa Bay Lightning',
  'team_abbrev': 'TBL',
  'logo_url': 'https://assets.nhle.com/logos/nhl/svg/TBL_light.svg'},
 {'division_name': 'Atlantic',
  'co

In [13]:
#Insertion Into SQL

In [5]:
import pymysql

conn = pymysql.connect(
    host="localhost",
    user="root",
    password="root",
    database="nhl",
)

cursor = conn.cursor()

In [6]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS teams (
        team_id INT AUTO_INCREMENT PRIMARY KEY,
        team_abbrev VARCHAR(10) UNIQUE,
        team_name VARCHAR(255),
        conference_name VARCHAR(255),
        division_name VARCHAR(255),
        logo_url TEXT
    )
""")

0

In [7]:
insert_query = """
    INSERT INTO teams (
        team_abbrev, team_name, conference_name, division_name, logo_url
    )
    VALUES (%s, %s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
        team_name = VALUES(team_name),
        conference_name = VALUES(conference_name),
        division_name = VALUES(division_name),
        logo_url = VALUES(logo_url)
"""

for team in team_records:
    value = (
        team["team_abbrev"],
        team["team_name"],
        team["conference_name"],
        team["division_name"],
        team["logo_url"],
    )
    cursor.execute(insert_query, value)

conn.commit()
print(f"Loaded {len(team_records)} teams into the teams table")

Loaded 32 teams into the teams table
